In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import os
import math

In [21]:
df = pd.read_csv("../results/surface_hour.csv")


memory_bins = [10, 14, 18, 22, 24, 24.5, 25]
memory_labels = [f"[{memory_bins[i]},{memory_bins[i+1]})" for i in range(len(memory_bins)-1)]

df['memory_bin'] = pd.cut(
    df['memory_window'],
    bins=memory_bins,
    labels=memory_labels,
    include_lowest=True,
    right=False
)


drift_bins = [0.005, 0.05, 0.2, 0.4, 0.8, 1.0]
drift_labels = [f"[{drift_bins[i]},{drift_bins[i+1]})" for i in range(len(drift_bins)-1)]

df['drift_bin'] = pd.cut(
    df['drift_scale'],
    bins=drift_bins,
    labels=drift_labels,
    include_lowest=True,
    right=False
)


grouped = df.groupby(['memory_bin', 'drift_bin'])['noise_scale'].max().reset_index()


pivot_table = grouped.pivot(index='memory_bin', columns='drift_bin', values='noise_scale')


pivot_table = pivot_table.iloc[::-1]


plt.figure(figsize=(10,8))
ax = sns.heatmap(
    pivot_table,
    annot=True,       
    fmt=".2f",        
    cmap="PRGn",    
    cbar_kws={"label":"Noise Scale Tolerance"},
    linewidths=0.5,
    linecolor="black"  
)
ax.figure.axes[-1].yaxis.label.set_size(15)
ax.figure.axes[-1].yaxis.label.set_weight("bold")

for text in ax.texts:
    text.set_fontsize(13)
    text.set_weight("bold")

ax.set_xlabel("Drift Scale", fontsize=15, weight="bold")
ax.set_ylabel(r"Memory Window($\mu$S)", fontsize=15, weight="bold")
ax.tick_params(axis="both", which="major", labelsize=15)
ax.set_title("1 HOUR", fontsize=15, weight="bold")

plt.tight_layout()
plt.savefig("./graph/3dheatmap/hour_table.png", dpi=300)
plt.show()

/tmp/ipykernel_16350/4167701627.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(['memory_bin', 'drift_bin'])['noise_scale'].max().reset_index()
